# Planwise: Annual Cost Calculator

Rechelle Transeth · November 15, 2026

This notebook calculates a **restricted prototype estimate** for the connected synthetic profile. It is not yet independently validated or a recommendation engine.

**Supported demonstration:** two reviewed oral generic medications, 30-, 60-, and 90-day in-area retail fills, no Extra Help or other payer contributions, and modeled annual gross drug spending below $2,100. Higher-spending profiles are flagged because qualifying annual spending requires additional plan rules.

Each medication starts January 1 and repeats at its selected interval: **13, 7, or 5 fills during 2026** for 30, 60, or 90 days, respectively. Unused supply can carry into the next year. Prices are frozen to the quarterly snapshot. See `docs/COST_ENGINE.md` for assumptions and sources.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'data/planwise.duckdb').exists(): ROOT=ROOT/'planwise_data'
assert (ROOT/'data/planwise.duckdb').exists(), 'Open from planwise_data with the prepared database.'
sys.path.insert(0,str(ROOT/'scripts'))
from cost_engine import compare_plans, json_default
from IPython.display import display
profile=json.loads((ROOT/'evidence/integrated_intake_example.json').read_text())
display(pd.DataFrame(profile['tier1']['medications']))
print(profile['tier1']['pharmacy_location']['location_name'])

,medication_name,rxcui,ndc,strength,dosage_form,quantity_per_fill,days_supply,assume_year_round_use
0,atorvastatin 20 MG Oral Tablet,617310,60505257908,20 mg,Oral Tablet,30.0,30,True
1,lisinopril 10 MG Oral Tablet,314076,68180098003,10 mg,Oral Tablet,30.0,30,True


KROGER CO OF MICHIGAN — 2600 POINTE TREMBLE RD, ALGONAC, MI 48001


## Plan comparisons
Annual medication expenses already include deductible payments. Annual total = annual premium + annual medication expenses. The rows are in plan-identifier order, not recommendation order. An unavailable estimate must not be treated as zero.

In [2]:
result=compare_plans(profile)
assert result['status']=='completed',result.get('errors')
columns=['plan_name','pharmacy_status','annual_premium','annual_medication_cost','deductible_paid','annual_total','status']
display(pd.DataFrame([{k:p.get(k) for k in columns} for p in result['plans']]))
(ROOT/'evidence/cost_engine_example.json').write_text(json.dumps(result,indent=2,default=json_default))
print('Plans with estimates:',sum(p['status']=='estimated_under_documented_assumptions' for p in result['plans']))

,plan_name,pharmacy_status,annual_premium,annual_medication_cost,deductible_paid,annual_total,status
0,Wellcare Classic (PDP),preferred,0.00,86.06,86.06,86.06,estimated_under_documented_assumptions
1,Wellcare Value Script (PDP),preferred,0.00,0.00,0,0.00,estimated_under_documented_assumptions
2,Prescription Blue Select (PDP),preferred,940.80,57.59,57.59,998.39,estimated_under_documented_assumptions
3,Prescription Blue Premium (PDP),preferred,1280.40,26.00,0,1306.40,estimated_under_documented_assumptions
4,SilverScript Choice (PDP),standard,939.60,72.54,72.54,1012.14,estimated_under_documented_assumptions
5,HealthSpring Assurance Rx (PDP),standard,0.00,42.38,42.38,42.38,estimated_under_documented_assumptions
6,HealthSpring Extra Rx (PDP),standard,604.80,69.81,0,674.61,estimated_under_documented_assumptions
7,Humana Basic Rx Plan (PDP),standard,0.00,69.29,69.29,69.29,estimated_under_documented_assumptions
8,Humana Premier Rx Plan (PDP),standard,1177.20,69.29,0,1246.49,estimated_under_documented_assumptions
9,Humana Value Rx Plan (PDP),standard,0.00,26.00,0,26.00,estimated_under_documented_assumptions


Plans with estimates: 12


## Follow one plan through the year
The first plan below is an example, not a selected recommendation. December can include two fills because the schedule is every 30 days, not once per calendar month.

In [3]:
example=next(p for p in result['plans'] if p['status']=='estimated_under_documented_assumptions')
print(example['plan_name'])
display(pd.DataFrame(example['medication_inputs'])[['medication_name','tier','unit_cost','dispensing_fee','gross_per_fill','deductible_applies']])
display(pd.DataFrame(example['monthly_costs']))
display(pd.DataFrame(example['fill_ledger']).head(6))

Wellcare Classic (PDP)


,medication_name,tier,unit_cost,dispensing_fee,gross_per_fill,deductible_applies
0,atorvastatin 20 MG Oral Tablet,1,0.147600,0.000000,4.43,True
1,lisinopril 10 MG Oral Tablet,1,0.073000,0.000000,2.19,True


,month,premium,medication_cost,total
0,1,0.00,13.24,13.24
1,2,0.00,0,0.00
2,3,0.00,6.62,6.62
3,4,0.00,6.62,6.62
4,5,0.00,13.24,13.24
5,6,0.00,6.62,6.62
6,7,0.00,6.62,6.62
7,8,0.00,6.62,6.62
8,9,0.00,6.62,6.62
9,10,0.00,6.62,6.62


,date,ndc,days_supply,medication_name,gross_cost,deductible_paid,post_deductible_payment,member_payment,deductible_remaining
0,2026-01-01,60505257908,30,atorvastatin 20 MG Oral Tablet,4.43,4.43,0,4.43,610.57
1,2026-01-01,68180098003,30,lisinopril 10 MG Oral Tablet,2.19,2.19,0,2.19,608.38
2,2026-01-31,60505257908,30,atorvastatin 20 MG Oral Tablet,4.43,4.43,0,4.43,603.95
3,2026-01-31,68180098003,30,lisinopril 10 MG Oral Tablet,2.19,2.19,0,2.19,601.76
4,2026-03-02,60505257908,30,atorvastatin 20 MG Oral Tablet,4.43,4.43,0,4.43,597.33
5,2026-03-02,68180098003,30,lisinopril 10 MG Oral Tablet,2.19,2.19,0,2.19,595.14


## Compare all three fill lengths
These examples assume one tablet per day, so quantity is explicitly 30, 60, or 90 tablets. Each run uses the matching supply-specific price, dispensing fee, and payment rule. Unavailable combinations remain visible; a 30-day rate is never substituted. Real profiles must supply their actual quantity, not an inferred dosage.

In [4]:
from copy import deepcopy
supply_rows=[]
for supply in (30,60,90):
    variant=deepcopy(profile)
    for medication in variant['tier1']['medications']:
        medication['days_supply']=supply
        medication['quantity_per_fill']=supply
    comparison=compare_plans(variant)
    (ROOT/f'evidence/cost_engine_{supply}_day_example.json').write_text(json.dumps(comparison,indent=2,default=json_default))
    for plan in comparison['plans']:
        supply_rows.append({'days_supply':supply,'plan_name':plan['plan_name'],'annual_total':plan['annual_total'],'status':plan['status'],'reason':'; '.join(plan['unavailable_reasons'])})
display(pd.DataFrame(supply_rows))

,days_supply,plan_name,annual_total,status,reason
0,30,Wellcare Classic (PDP),86.06,estimated_under_documented_assumptions,
1,30,Wellcare Value Script (PDP),0.00,estimated_under_documented_assumptions,
2,30,Prescription Blue Select (PDP),998.39,estimated_under_documented_assumptions,
3,30,Prescription Blue Premium (PDP),1306.40,estimated_under_documented_assumptions,
4,30,SilverScript Choice (PDP),1012.14,estimated_under_documented_assumptions,
5,30,HealthSpring Assurance Rx (PDP),42.38,estimated_under_documented_assumptions,
6,30,HealthSpring Extra Rx (PDP),674.61,estimated_under_documented_assumptions,
7,30,Humana Basic Rx Plan (PDP),69.29,estimated_under_documented_assumptions,
8,30,Humana Premier Rx Plan (PDP),1246.49,estimated_under_documented_assumptions,
9,30,Humana Value Rx Plan (PDP),26.00,estimated_under_documented_assumptions,


## Unsupported inputs stay visible
This example changes a fill to an unsupported 45 days. The calculator rejects it rather than converting the supply or inventing a payment.

In [5]:
from copy import deepcopy
unsupported=deepcopy(profile)
unsupported['tier1']['medications'][0]['days_supply']=45
blocked=compare_plans(unsupported)
print(blocked['status'])
print(*blocked['errors'],sep='\n')

invalid_or_unsupported_input
Medication 1: This prototype currently maps 30, 60 or 90 days only.
Only 30-, 60-, and 90-day retail fills are supported


## Development checks
Eight reproducible scenarios exercise the connection, including mixed fill lengths and inputs that need clarification. These variations use the same two reviewed medications; they are not a representative population or the final evaluation set.

The completeness counts show when some plans cannot be estimated. Missing estimates are never treated as inexpensive plans. Peak monthly spending is a modeled amount under the fixed fill schedule, not a promise of predictable real-world costs.

In [6]:
from development_profiles import build_profiles
from cost_engine import compare_profiles
profiles=build_profiles()
batch=compare_profiles(profiles)
display(pd.DataFrame(batch['summary']))
(ROOT/'evidence/cost_engine_development_profiles.json').write_text(json.dumps(profiles,indent=2))
(ROOT/'evidence/cost_engine_batch_results.json').write_text(json.dumps(batch,indent=2,default=json_default))
mixed=batch['results']['mixed_30_and_90_days']
display(pd.DataFrame([{k:p.get(k) for k in ['plan_name','annual_total','fill_count','average_monthly_total','peak_monthly_total','issue_codes']} for p in mixed['plans']]))
print('Mixed-fill completeness:',mixed['coverage_summary'])

,profile,status,plans_estimated,plans_unavailable,input_errors
0,two_medications_30_days,completed,12,0,[]
1,two_medications_60_days,completed,5,7,[]
2,two_medications_90_days,completed,12,0,[]
3,mixed_30_and_90_days,completed,12,0,[]
4,one_medication,completed,12,0,[]
5,no_medications,completed,12,0,[]
6,clarification_needed,invalid_or_unsupported_input,0,0,[Medication 1: strength does not match the sel...
7,outside_subsidy_scope,invalid_or_unsupported_input,0,0,[Only profiles without Extra Help are supported]


,plan_name,annual_total,fill_count,average_monthly_total,peak_monthly_total,issue_codes
0,Wellcare Classic (PDP),92.34,18,7.70,15.81,[]
1,Wellcare Value Script (PDP),0.00,18,0.00,0.00,[]
2,Prescription Blue Select (PDP),997.79,18,83.15,88.07,[]
3,Prescription Blue Premium (PDP),1303.40,18,108.62,110.70,[]
4,SilverScript Choice (PDP),1017.51,18,84.79,92.43,[]
5,HealthSpring Assurance Rx (PDP),43.48,18,3.62,7.64,[]
6,HealthSpring Extra Rx (PDP),677.52,18,56.46,63.27,[]
7,Humana Basic Rx Plan (PDP),68.44,18,5.70,11.72,[]
8,Humana Premier Rx Plan (PDP),1245.64,18,103.80,109.82,[]
9,Humana Value Rx Plan (PDP),28.00,18,2.33,5.00,[]


Mixed-fill completeness: {'plans_considered': 12, 'plans_estimated': 12, 'plans_unavailable': 0, 'complete_comparison': True}


## Validation and remaining work
The checks below test arithmetic and data integration. They do not establish agreement with real pharmacy quotes or Medicare Plan Finder.

Remaining work: independently validate price construction and deductible-crossing claims; implement qualifying annual-spending rules for higher-cost profiles; review additional medication products. Insulin, vaccines, mail order, and subsidized estimates are not enabled.

In [7]:
import unittest, io
suite=unittest.defaultTestLoader.discover(str(ROOT/'tests'),pattern='test_cost_engine.py')
stream=io.StringIO()
run=unittest.TextTestRunner(stream=stream,verbosity=1).run(suite)
print(stream.getvalue())
assert run.wasSuccessful()
checks={'tests_run':run.testsRun,'failures':len(run.failures),'errors':len(run.errors),'external_validation':False}
(ROOT/'evidence/cost_engine_checks.json').write_text(json.dumps(checks,indent=2))

.............................
----------------------------------------------------------------------
Ran 29 tests in 5.794s

OK



85